# 62 — Emissions Documentary Layer

Normalises `emissions_long.parquet` into a site-year-pollutant summary layer.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "62_emissions_documentary_layer"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

emissions, meta = safe_read_parquet(DATA / "emissions_long.parquet")
if emissions.empty:
    raise FileNotFoundError("data_sources/emissions_long.parquet missing or empty")

cfg = load_yaml(CONFIGS / "documentary_sources.yml")
possible_site_cols = cfg.get("site_columns_preference", ["site_name", "site", "facility", "operator_site"])
possible_year_cols = cfg.get("year_columns_preference", ["year", "report_year"])
possible_pollutant_cols = cfg.get("pollutant_columns_preference", ["pollutant", "parameter", "analyte"])
possible_value_cols = cfg.get("value_columns_preference", ["value", "result", "measured_value", "annual_mean"])
possible_unit_cols = cfg.get("unit_columns_preference", ["unit", "units"])
possible_limit_cols = cfg.get("limit_columns_preference", ["limit", "permit_limit", "elv", "emission_limit"])

def pick(cols):
    for c in cols:
        if c in emissions.columns:
            return c
    return None

site_col = pick(possible_site_cols)
year_col = pick(possible_year_cols)
pollutant_col = pick(possible_pollutant_cols)
value_col = pick(possible_value_cols)
unit_col = pick(possible_unit_cols)
limit_col = pick(possible_limit_cols)

if not site_col or not year_col or not pollutant_col:
    raise RuntimeError(f"Could not resolve key columns. site={site_col}, year={year_col}, pollutant={pollutant_col}")

work = emissions.copy()
work["site_name"] = work[site_col].astype(str).str.strip()
work["site_key"] = work["site_name"].map(slugify)
work["report_year"] = pd.to_numeric(work[year_col], errors="coerce")
work["pollutant_std"] = work[pollutant_col].map(standardise_pollutant)
work["value_num"] = pd.to_numeric(work[value_col], errors="coerce") if value_col else np.nan
work["limit_num"] = pd.to_numeric(work[limit_col], errors="coerce") if limit_col else np.nan
work["unit_std"] = work[unit_col].astype(str).str.strip() if unit_col else ""

summary = (
    work.groupby(["site_key", "site_name", "report_year", "pollutant_std"], as_index=False)
    .agg(
        rows=("pollutant_std", "size"),
        value_mean=("value_num", "mean"),
        value_max=("value_num", "max"),
        limit_mean=("limit_num", "mean"),
        unit_mode=("unit_std", lambda x: x.dropna().iloc[0] if len(x.dropna()) else "")
    )
    .sort_values(["site_name", "report_year", "pollutant_std"])
)

summary["limit_exceedance_flag"] = np.where(
    pd.notna(summary["limit_mean"]) & pd.notna(summary["value_max"]) & (summary["value_max"] > summary["limit_mean"]),
    1, 0
)

summary.to_csv(out / "site_year_pollutant_summary.csv", index=False)
write_json(out / "emissions_documentary_summary.json", {
    "rows_in": int(len(work)),
    "rows_out": int(len(summary)),
    "sites": int(summary["site_key"].nunique()) if not summary.empty else 0,
    "years": int(summary["report_year"].nunique()) if not summary.empty else 0,
    "pollutants": int(summary["pollutant_std"].nunique()) if not summary.empty else 0,
})

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml"])
manifest["inputs"] = {"emissions_long": meta}
add_artifact(manifest, out / "site_year_pollutant_summary.csv")
add_artifact(manifest, out / "emissions_documentary_summary.json")
write_json(out / "manifest.json", manifest)

display(summary.head(20))